In [1]:
# Cell 1: Install & Load Online Retail Dataset directly from UCI Repository
!pip install -q ucimlrepo
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

# Fetch Online Retail Dataset (ID: 352)
online_retail = fetch_ucirepo(id=352)
df_raw = online_retail.data.original.copy()

print("Raw Dataset Shape:", df_raw.shape)
print("\nColumns:", df_raw.columns.tolist())
df_raw.head()

Raw Dataset Shape: (541909, 8)

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [2]:
# Cell 2: Data Quality Audit & Cleaning (Handling Nulls, Duplicates, Refunds)
df = df_raw.copy()

# 1. Standardize column names & data types
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

# 2. Capture Data Quality Metrics BEFORE cleaning
total_raw_rows = len(df)
null_cust_rows = df['CustomerID'].isna().sum()
duplicate_rows = df.duplicated().sum()
refund_rows = (df['InvoiceNo'].str.startswith('C') | (df['Quantity'] <= 0)).sum()
invalid_price_rows = (df['UnitPrice'] <= 0).sum()

print("=== DATA QUALITY AUDIT (BEFORE CLEANING) ===")
print(f"Total Raw Rows:           {total_raw_rows:,}")
print(f"Missing CustomerID Rows:  {null_cust_rows:,}")
print(f"Exact Duplicate Rows:     {duplicate_rows:,}")
print(f"Refund / Neg Qty Rows:    {refund_rows:,}")
print(f"Zero/Neg UnitPrice Rows:  {invalid_price_rows:,}")

# 3. Save refunds separately (Edge case tracking)
df_refunds = df[df['InvoiceNo'].str.startswith('C') | (df['Quantity'] <= 0)].copy()

# 4. Clean dataset for RFM Analysis
df_clean = df.drop_duplicates().copy()
df_clean = df_clean.dropna(subset=['CustomerID'])
df_clean = df_clean[~df_clean['InvoiceNo'].str.startswith('C')]
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

# Format CustomerID & calculate TotalAmount
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

print("\n=== AFTER CLEANING ===")
print(f"Clean Valid Rows:         {len(df_clean):,}")
print(f"Unique Customers:         {df_clean['CustomerID'].nunique():,}")
print(f"Unique Invoices:          {df_clean['InvoiceNo'].nunique():,}")
print(f"Total Clean Revenue:      {df_clean['TotalAmount'].sum():,.2f}")
print(f"Date Range:               {df_clean['InvoiceDate'].min()} to {df_clean['InvoiceDate'].max()}")

# Export cleaned transactions CSV
df_clean.to_csv('cleaned_transactions.csv', index=False)
print("\nSaved: cleaned_transactions.csv")

=== DATA QUALITY AUDIT (BEFORE CLEANING) ===
Total Raw Rows:           541,909
Missing CustomerID Rows:  135,080
Exact Duplicate Rows:     5,268
Refund / Neg Qty Rows:    10,624
Zero/Neg UnitPrice Rows:  2,517

=== AFTER CLEANING ===
Clean Valid Rows:         392,692
Unique Customers:         4,338
Unique Invoices:          18,532
Total Clean Revenue:      8,887,208.89
Date Range:               2010-12-01 08:26:00 to 2011-12-09 12:50:00

Saved: cleaned_transactions.csv
